## **Data Science Aplicado a las Finanzas** 🚀
### **HW Sesión 8**

Andrés C. Medina Sanhueza

Senior Datascientist Engineer

anmedinas@gmail.com

Consideraciones Previas

* Tarea es Individual, debe subir al repositorio personal (notebooks)

* Fecha Entrega : Viernes 10 de Julio 23:59 (se recomienda hacer `commits` cada vez que hagan cambios o subir, esto, también es parte de la evaluación, es decir, trazabilidad del trabajo).

---

## 1. Fundamentos Teóricos: Estacionariedad, Persistencia y Momentos en GARCH(1,1)

Considere el modelo GARCH(1,1):

\begin{equation*}
\begin{aligned}
r_t &= \mu + \epsilon_t \\
\epsilon_t &= \sigma_t z_t, \quad z_t \overset{iid}{\sim} \mathcal{N}(0,1) \\
\sigma_t^2 &= \omega + \alpha\,\epsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2
\end{aligned}
\end{equation*}

con $\omega>0$, $\alpha, \beta \geq 0$.

### 1.1 Estacionariedad en Covarianza

**(a)** Muestre que, escribiendo $\sigma_t^2$ como un proceso ARMA(1,1) en $\epsilon_t^2$ (representación ARCH($\infty$)), la condición necesaria y suficiente para que el proceso sea débilmente estacionario es $\alpha+\beta<1$.

**(b)** Bajo dicha condición, derive la varianza incondicional:
$$\sigma^2 = \mathbb{E}[\sigma_t^2] = \frac{\omega}{1-\alpha-\beta}$$

**(c)** ¿Qué ocurre con $\sigma^2$ cuando $\alpha+\beta \to 1$? Interprete este límite en términos del modelo `IGARCH` (Integrated GARCH).

### 1.2 Persistencia y Vida Media de un Shock

Un shock de volatilidad en $t$ decae geométricamente a tasa $(\alpha+\beta)$.

**(a)** Muestre que $\mathbb{E}[\sigma_{t+h}^2 - \sigma^2 \mid \mathcal{F}_t] = (\alpha+\beta)^h(\sigma_{t+1}^2-\sigma^2)$.

**(b)** Defina la **vida media** (*half-life*) del shock como el número de períodos $h^{*}$ tal que la desviación respecto a $\sigma^2$ se reduce a la mitad. Muestre que:
$$h^{*} = \frac{\ln(0.5)}{\ln(\alpha+\beta)}$$

**(c)** Calcule $h^{*}$ para $(\alpha,\beta) = (0.05, 0.90)$, $(0.10, 0.85)$ y $(0.02, 0.97)$. ¿Qué combinación implica volatilidad más persistente?

### 1.3 Momento de Cuarto Orden y Curtosis

Asumiendo $z_t \sim \mathcal{N}(0,1)$ y $3\alpha^2+2\alpha\beta+\beta^2<1$ (condición para que exista el cuarto momento incondicional):

**(a)** Muestre que la curtosis incondicional de $\epsilon_t$ es:
$$\kappa = \frac{3\big[1-(\alpha+\beta)^2\big]}{1-(\alpha+\beta)^2-2\alpha^2}$$

**(b)** Muestre que $\kappa > 3$ siempre que $\alpha>0$, es decir, que el modelo GARCH(1,1) con innovaciones normales genera **exceso de curtosis** incondicional (leptocurtosis) aunque $z_t$ sea gaussiano. Relacione este resultado con los hechos estilizados vistos en clases.

**(c)** Verifique la condición de existencia del cuarto momento para los tres pares $(\alpha,\beta)$ del punto 1.2(c). ¿Existe algún caso en que la curtosis no esté definida?

## 2. Simulación de Modelos de Volatilidad

Simule 3.000 observaciones ($r_t$) con $\mu=0$ para cada uno de los siguientes procesos, descartando las primeras 500 observaciones como *burn-in*:

* **ARCH(1)**: $\omega=0.05$, $\alpha_1=0.85$
* **GARCH(1,1)** — baja persistencia: $\omega=0.05$, $\alpha=0.05$, $\beta=0.60$
* **GARCH(1,1)** — alta persistencia: $\omega=0.02$, $\alpha=0.08$, $\beta=0.90$
* **EGARCH(1,1)**: $\omega=0.01$, $\alpha=0.10$, $\gamma=-0.08$, $\beta=0.95$
* **GJR-GARCH(1,1)**: $\omega=0.02$, $\alpha=0.03$, $\gamma=0.09$, $\beta=0.90$

Para cada proceso:

**(a)** Implemente la recursión correspondiente en Python **sin usar la librería `arch`** (puede reutilizar y adaptar las funciones `simulate_arch_model`/`simulate_garch_model` vistas en clases, extendiéndolas a EGARCH y GJR-GARCH).

**(b)** Grafique en un mismo panel la serie simulada $r_t$ y su volatilidad condicional $\sigma_t$.

**(c)** Para los dos GARCH(1,1) simulados, calcule la varianza muestral de $\epsilon_t^2$ sobre las 2.500 observaciones post *burn-in* y compárela con la varianza incondicional teórica $\omega/(1-\alpha-\beta)$ derivada en 1.1(b). Comente la calidad de la aproximación y cómo cambia si aumenta el número de simulaciones a 50.000.

**(d)** Compare visualmente la asimetría del EGARCH y del GJR-GARCH: simule choques $z_t$ idénticos (misma semilla) para ambos modelos y grafique $\sigma_t$ resultante ante una racha de choques negativos vs. una racha de choques positivos de igual magnitud. ¿Qué modelo reacciona más fuerte a los choques negativos?

## 3. Estimación por Máxima Verosimilitud de un GARCH(1,1) Programada desde Cero

Use la serie **GARCH(1,1) de alta persistencia** simulada en el Ejercicio 2 ($\omega=0.02$, $\alpha=0.08$, $\beta=0.90$).

### 3.1 Log-Verosimilitud Condicional

Bajo $z_t \sim \mathcal{N}(0,1)$, la contribución de cada observación a la log-verosimilitud es:
$$\ell_t(\boldsymbol{\psi}) = -\frac{1}{2}\Big[\ln(2\pi) + \ln(\sigma_t^2) + \frac{\epsilon_t^2}{\sigma_t^2}\Big]$$

con $\boldsymbol{\psi}=(\omega,\alpha,\beta)^\top$ y $\sigma_1^2$ inicializado en la varianza muestral incondicional de $\epsilon_t$.

**(a)** Implemente `compute_conditional_variance(psi, eps)` que recupere $\sigma_t^2$ recursivamente.

**(b)** Implemente `neg_log_likelihood(psi, eps)` que retorne $-\sum_{t=1}^{T}\ell_t(\boldsymbol{\psi})$.

**(c)** Reparametrice para imponer $\omega>0$, $\alpha,\beta\geq0$ y $\alpha+\beta<1$ (por ejemplo, optimizando sobre $\ln\omega$ y una transformación logística para $\alpha,\beta$), o use `scipy.optimize.minimize` con `method='SLSQP'` y restricciones explícitas.

**(d)** Minimice partiendo de $\boldsymbol{\psi}_0=(0.01,\ 0.05,\ 0.85)^\top$. Reporte $\hat{\boldsymbol{\psi}}$.

> **Restricción:** No está permitido usar la librería `arch` en esta sección.

### 3.2 Validación con `arch`

Estime el mismo modelo con `arch_model(eps, vol='GARCH', p=1, q=1, dist='normal')` y complete la tabla:

| Parámetro | Verdadero | MLE propio | `arch` |
|-----------|-----------|------------|--------|
| $\omega$  | 0.02 | | |
| $\alpha$  | 0.08 | | |
| $\beta$   | 0.90 | | |
| Log-Lik.  | —    | | |

Explique cualquier diferencia observada (inicialización de $\sigma_1^2$, algoritmo de optimización, escala de los datos, etc.).

## 4. Aplicación a Datos Reales: Hechos Estilizados y Selección de Modelo

Elija **dos activos** de distinta naturaleza (por ejemplo, una acción individual y un índice, o un activo desarrollado y uno emergente/cripto) y descargue sus precios de cierre diarios con `yfinance` para el período **enero 2015 – diciembre 2024**. Calcule los retornos logarítmicos porcentuales ($r_t = 100\times\Delta\ln P_t$).

### 4.1 Hechos Estilizados

Para cada activo:

1. Grafique la serie de precios y de retornos.
2. Calcule *skewness* y *kurtosis* muestral de los retornos. Compárela con la normal (skew=0, kurt=3) y con la curtosis teórica derivada en 1.3(a) evaluada en los parámetros que estime en 4.2.
3. Grafique el histograma de retornos junto a una densidad normal y una $t$-Student ajustadas. ¿Cuál se ajusta mejor a las colas?
4. Grafique la volatilidad móvil (ventana de 20 días) para evidenciar *clustering*.
5. Calcule la correlación entre $r_{t-1}$ y $r_t^2$ (proxy simple del efecto apalancamiento). ¿El signo es consistente con lo esperado?

### 4.2 Selección del Modelo: Grilla de Especificaciones

Para cada activo, ajuste **los cuatro modelos** vistos en clases (`ARCH`, `GARCH`, `EGARCH`, `GJR-GARCH`, todos con $p=q=1$, y $o=1$ donde corresponda) cruzados con **tres distribuciones** (`normal`, `t`, `skewt`) — 12 modelos por activo.

1. Construya una tabla con AIC y BIC de los 12 modelos, ordenada por BIC.
2. Seleccione el mejor modelo por activo. ¿Coincide la mejor especificación entre AIC y BIC? ¿Coincide entre los dos activos?
3. Reporte los coeficientes estimados del modelo ganador (incluyendo el parámetro de asimetría/forma de la distribución si corresponde) con sus errores estándar.
4. Calcule la persistencia $\hat\alpha+\hat\beta$ (o su equivalente para EGARCH/GJR-GARCH) del modelo ganador de cada activo y la vida media asociada (fórmula de 1.2(b)). ¿Qué activo tiene volatilidad más persistente?

## 5. Diagnóstico del Modelo Ganador

Para el modelo seleccionado en 4.2 de **cada activo**, sobre los residuos estandarizados $\hat z_t = \hat\epsilon_t/\hat\sigma_t$:

1. Grafique $\hat z_t$ en el tiempo y su ACF/PACF (20 rezagos).
2. Grafique la ACF de $\hat z_t^2$ (20 rezagos). ¿Queda heterocedasticidad residual?
3. Aplique el test de **Ljung-Box** sobre $\hat z_t$ y sobre $\hat z_t^2$, con $k=10$ y $k=20$. Reporte estadístico y $p$-value para cada caso.
4. Aplique el test **ARCH-LM** sobre $\hat z_t$. ¿Se rechaza la ausencia de efectos ARCH residuales?
5. Evalúe la distribución de $\hat z_t$: histograma vs. densidad ajustada, Q-Q plot contra la distribución supuesta en el modelo (normal, $t$ o $skew$-$t$ según corresponda) y test de **Jarque-Bera**.
6. Con base en 1–5, concluya si el modelo ganador captura adecuadamente la dinámica de volatilidad. Si detecta problemas, proponga (sin necesariamente implementar) una especificación alternativa.

## 6. Pronóstico y Gestión de Riesgo: Value-at-Risk

Trabaje con el modelo ganador de **un** activo (elija el que le resulte más interesante).

### 6.1 Pronóstico de Volatilidad

**(a)** Con el modelo ajustado sobre toda la muestra, genere un pronóstico de $\sigma_t^2$ a 10 días (`horizon=10`) y grafique la trayectoria pronosticada junto a la volatilidad condicional histórica de los últimos 60 días.

**(b)** Calcule la volatilidad de largo plazo implícita del modelo ($\sigma^2$ de 1.1(b) o su equivalente) y compárela con el nivel al que converge el pronóstico.

### 6.2 VaR Paramétrico *Rolling* y Backtesting

Reserve los **últimos 250 días** de la muestra como ventana de evaluación *fuera de muestra*. Para cada día $t$ en esa ventana:

**(a)**

1. Reestime (o utilice una aproximación *rolling*/*expanding window*, a su elección, pero sea explícito y consistente) el modelo ganador usando información hasta $t-1$.
2. Pronostique $\hat\sigma_t$ a un paso.
3. Calcule el **VaR paramétrico al 99%** para el retorno del día $t$:
$$\widehat{VaR}_t^{0.99} = -\big(\hat\mu + \hat\sigma_t\, q_{0.01}\big)$$
donde $q_{0.01}$ es el cuantil 1% de la distribución de innovaciones supuesta por el modelo (normal, $t$ o $skew$-$t$).
4. Grafique los retornos reales junto a $-\widehat{VaR}_t^{0.99}$ e identifique visualmente las violaciones ($r_t < -\widehat{VaR}_t^{0.99}$).

**(b)** *Backtesting* — Test de Kupiec (*Proportion of Failures*):

Sea $N$ el número de violaciones observadas en $T=250$ días y $p=0.01$ la tasa esperada. El estadístico de razón de verosimilitud es:
$$LR_{POF} = -2\ln\left[\frac{(1-p)^{T-N}p^{N}}{(1-\hat p)^{T-N}\hat p^{N}}\right] \sim \chi^2_{(1)}, \qquad \hat p = N/T$$

Calcule $LR_{POF}$ y su $p$-value. ¿Rechaza, al 5%, que el modelo tiene una tasa de cobertura correcta?

**(c)** Repita el cálculo del VaR usando **simplemente la volatilidad no condicional histórica** (desviación estándar muestral de toda la ventana de entrenamiento, constante en el tiempo) como *benchmark* ingenuo. Compare el número de violaciones y el resultado del test de Kupiec frente al modelo GARCH/EGARCH/GJR-GARCH. ¿El modelo condicional mejora la cobertura de riesgo respecto al benchmark?

## 7. Discusión Final

Responda de forma breve y argumentada (máx. media página por pregunta):

1. De los cuatro modelos evaluados (ARCH, GARCH, EGARCH, GJR-GARCH), ¿cuál domina de forma consistente entre sus dos activos? ¿Es razonable esperar que la misma especificación funcione bien para activos de distinta naturaleza?

2. En base a los resultados de la Sección 1.3 y 4.1, discuta la relación entre la curtosis incondicional teórica del modelo GARCH normal y la curtosis empírica observada en los datos. ¿Es suficiente asumir innovaciones gaussianas para capturar las colas pesadas observadas, o se requiere una distribución $t$/$skew$-$t$?

3. ¿Qué limitaciones tiene el enfoque de VaR paramétrico *univariado* implementado en la Sección 6 para un banco o fondo que gestiona una cartera de múltiples activos? Mencione al menos dos extensiones (por ejemplo, modelos multivariados de volatilidad, `DCC-GARCH`, *Filtered Historical Simulation*, medidas de riesgo como *Expected Shortfall*) y explique brevemente qué problema resuelven.

⚠️ **Énfasis:** Use exclusivamente las herramientas vistas en sesión 8 (`arch`, ARCH/GARCH/EGARCH/GJR-GARCH, Ljung-Box, ARCH-LM, Jarque-Bera) salvo en la Sección 3, donde la implementación debe ser propia. La justificación económica/estadística de cada resultado tiene el mismo peso que la implementación técnica.